# FAIDR — interactive notebook

![Schematic of the FAIDR statistical model.](FAIDR_scheme.jpeg)

*Schematic of the FAIDR statistical model.*

This notebook runs FAIDR pipeline. Publication: https://doi.org/10.7554/eLife.60220


> **Runtime:** with default settings (100 repetitions) expect **tens of minutes**. Two progress bars show the current stage.

## FAIDR — a brief introduction to the algorithm

FAIDR learns which IDRs are characteristic of a given functional category or protein set (e.g. a condensate, a GO term, or a disease-gene list). As a result, a positive prediction can be interpreted as:

"This IDR looks similar to IDRs associated with this condensate or functional category".

An important point is that the model is not simply memorizing the original labels.

Instead, it uses an expectation-maximisation (EM) algorithm to handle the fact that annotations are available only at the protein level while predictions are made at the IDR level.

As a result:

- some IDRs from proteins not in the annotation set may receive high predicted probabilities if their features strongly resemble those of annotated IDRs;
- conversely, some IDRs from annotated proteins may receive lower scores if their feature profiles do not match the overall learned pattern.

This behavior is expected and is one of the intended goals of the approach:

1. the experimental labels are available only at the protein level. But many proteins contain multiple IDRs. Not all of these IDRs are necessarily responsible for the protein's association with that functional category. Consequently, some IDRs may carry little or no signal related to the function.
2. the protein-level annotation does not specify *which* IDR within a positive protein is responsible for the function. The EM algorithm estimates this as a hidden variable: at each iteration it up-weights IDRs whose features best explain the annotation and down-weights the others. IDRs in positive proteins whose features do not match the learned pattern therefore receive lower predicted probabilities — not because the protein label is wrong, but because they are unlikely to be the functional IDR.

## Environment & packages

Choose **one** of the three options below. Run the cells in order within each option.

---

### Option A — directly from this notebook (pip, simplest)

> R ≥ 4.2 must be installed on your machine first:
> macOS `brew install r` · Linux `sudo apt install r-base` · Windows https://cran.r-project.org

1. Run the **"Install Python packages"** cell immediately below.
2. **Restart the kernel** (Kernel → Restart Kernel, or the ↺ button).
3. Run the **"Install R packages"** cell (next section).
4. Continue from **"Imports"** onwards.

---

### Option B — pip in terminal (one-time)

```bash
cd FAIDR_pretty
python3 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\activate
pip install -r requirements.txt
python -m ipykernel install --user --name=faidr --display-name "FAIDR (.venv)"
```

Select kernel **FAIDR (.venv)**, then skip to **"Install R packages"** below.

**Python version:** 3.8–3.12 (`rpy2` is not compatible with 3.13+).

---

### Option C — conda (handles R + Python together, most reliable)

```bash
conda env create -f environment.yml
conda activate faidr
python -m ipykernel install --user --name=faidr --display-name "FAIDR (conda)"
```

Select kernel **FAIDR (conda)**, then skip to **"Imports"** — R packages are already installed by conda.

In [ ]:
# Option A — Install Python packages directly from this notebook.
# Skip this cell if you used Option B or C above.
# After this cell finishes, RESTART THE KERNEL, then run the next cells.
import subprocess, sys

vi = sys.version_info
if vi < (3, 8) or vi >= (3, 13):
    raise RuntimeError(
        f"Python {vi.major}.{vi.minor} is not supported.\n"
        "FAIDR requires Python 3.8–3.12 (rpy2 is not compatible with 3.13+).\n"
        "Please select a supported kernel and re-run this cell."
    )

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt"],
    capture_output=True, text=True
)
lines = (result.stdout + result.stderr).strip().splitlines()
print('\n'.join(lines[-15:]))
if result.returncode == 0:
    print('\n--- Done. RESTART THE KERNEL now, then continue. ---')
else:
    raise RuntimeError('pip install failed — see output above.')


## Install required R packages

In [ ]:
import rpy2.robjects as ro

cran = "https://cloud.r-project.org"
for pkg in ("glmnet", "pROC"):
    ro.r(
        f'if (!requireNamespace("{pkg}", quietly=TRUE)) '
        f'install.packages("{pkg}", repos="{cran}")'
    )
print("Done — glmnet and pROC are installed (or were already present).")

In [ ]:
# Preflight check — confirms the environment is fully set up.
import importlib, sys

REQUIRED = ["rpy2", "pandas", "numpy", "matplotlib", "matplotlib_venn", "tqdm"]
missing = [p for p in REQUIRED if importlib.util.find_spec(p) is None]

if missing:
    raise ImportError(
        f"Missing packages: {', '.join(missing)}\n"
        "Run the 'Install Python packages' cell above, restart the kernel, "
        "then re-run this cell."
    )

print(f"Python {sys.version.split()[0]} — all required packages found. Ready to continue.")

## Imports for analysis


In [ ]:
%matplotlib inline

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

import importlib

import FAIDR_cv_notebook
importlib.reload(FAIDR_cv_notebook)
from FAIDR_cv_notebook import FEATURE_DEFAULTS, run_faidr_cv, uniprot_from_idr

In [ ]:
# Publication figure style — PNAS defaults.
# Run once after the imports cell; affects all subsequent plots.
import matplotlib as mpl

PNAS_SINGLE = 3.42   # single-column width, inches
PNAS_15COL  = 4.49   # 1.5-column width, inches
PNAS_DOUBLE = 7.01   # double-column width, inches

mpl.rcParams.update({
    # Font — Arial is PNAS standard; falls back to Helvetica / DejaVu Sans
    'font.family':        'sans-serif',
    'font.sans-serif':    ['Arial', 'Helvetica', 'Liberation Sans', 'DejaVu Sans'],
    'font.size':          8,
    'axes.titlesize':     9,
    'axes.titleweight':   'bold',
    'axes.labelsize':     8,
    'xtick.labelsize':    7,
    'ytick.labelsize':    7,
    'legend.fontsize':    7,
    'legend.frameon':     False,

    # Figure defaults
    'figure.figsize':     (PNAS_SINGLE, PNAS_SINGLE * 0.85),
    'figure.dpi':         150,
    'savefig.dpi':        300,
    'savefig.bbox':       'tight',
    'savefig.pad_inches': 0.02,

    # Axes — no top/right spines
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.linewidth':     0.75,
    'axes.labelpad':      4,

    # Ticks
    'xtick.major.width':  0.75,
    'ytick.major.width':  0.75,
    'xtick.major.size':   3.5,
    'ytick.major.size':   3.5,
    'xtick.direction':    'out',
    'ytick.direction':    'out',

    # Lines
    'lines.linewidth':    1.0,
    'patch.linewidth':    0.75,

    # Color cycle (ColorBrewer, colorblind-safe)
    'axes.prop_cycle': mpl.cycler(color=[
        '#2166ac',  # blue
        '#d6604d',  # red
        '#4dac26',  # green
        '#8073ac',  # purple
        '#e08214',  # orange
        '#01665e',  # teal
    ]),
})

print('Publication style applied — PNAS single-column defaults.')
print(f'Column widths available: PNAS_SINGLE={PNAS_SINGLE}", PNAS_15COL={PNAS_15COL}", PNAS_DOUBLE={PNAS_DOUBLE}"')


## Step 0 — Create a UniProt accession file

Prepare a .tsv file and add it to  ./uniprot_accessions folder. The file should contain the UniProt accessions for your compartment/functional category. See ./uniprot_accessions/GO0005730.tsv for an example.


Set the path to your file below:

In [ ]:
uniprot_list_file = "uniprot_accessions/GO0005730.tsv"

## Step 1 — Choose the feature set

FAIDR is trained using IDR-level labels (0/1) rather than protein-level labels. To create these labels, the proteins in your UniProt accession list are mapped to the IDRs.

IDRs from proteins in your UniProt list are assigned label 1.
All other IDRs are assigned label 0.

Because each feature dataset contains its own set of IDRs (and therefore its own IDR boundaries), you must first choose which feature dataset to use. Available datasets are located in feature_matrices/.:


| `feature_type` | File | Source |
|----------------|------|------------|
| `"evolutionary"` | `feature_matrices/HUMAN_ES.txt` | `https://www.pnas.org/doi/10.1073/pnas.2604562123` |
| `"signature"` | `feature_matrices/FS_UP000005640_9606_SPOTD_MIN_30AA.txt` | `https://www.pnas.org/doi/10.1073/pnas.2604562123` |
| `"ruff_et_al"` | `feature_matrices/ruff_et_al_features.tsv` | `https://www.cell.com/cell/fulltext/S0092-8674(25)01191-2`  |


### Custom feature file (optional)

To use your own features, set `features_path` to a tab-separated file (`.tsv` or `.txt`).

**Requirements:**
* One IDR per row
* First column: `idr_name`
* Remaining columns: numeric features

`idr_name` must start with a UniProt accession followed by an underscore, for example:
```text
Q5VUJ6_IDR_1_54
Q8TD19_IDR_825_961
```
FAIDR uses the accession (the part before the first `_`) to map proteins to IDRs. Missing values are treated as 0.


Set the corresponding value in the `feature_type` parameter below:


In [ ]:
feature_type = "evolutionary"
# feature_type = "signature"
# feature_type = "ruff_et_al"
features_path = None            # Set `features_path = None` to use a built-in dataset.

## Step 2 — Build IDR-level annotation from your Uniprot accession protein list


The cell below reads all IDRs from the feature matrix chosen in Step 1, assigns label **1** to IDRs whose protein is in your list, and **0** to all others. It also reports how many proteins from your list have at least one IDR in that feature file.

In [ ]:
category_name = Path(uniprot_list_file).stem
target_file = Path("annotation_targets") / f"{category_name}.tsv"

# --- load feature IDR names ---
features_file = Path(features_path) if features_path else Path(FEATURE_DEFAULTS[feature_type])
features_head = pd.read_csv(features_file, sep="\t", nrows=0)
if "idr_name" not in features_head.columns:
    raise ValueError(f"{features_file}: expected idr_name column")

idr_names = pd.read_csv(features_file, sep="\t", usecols=["idr_name"])["idr_name"].astype(str)
idr_df = pd.DataFrame({"idr_name": idr_names})
idr_df["uniprot_accession"] = idr_df["idr_name"].map(uniprot_from_idr)

# --- load user's protein list ---
uni_df = pd.read_csv(uniprot_list_file, sep="\t")
accession_col = "uniprot_accession" if "uniprot_accession" in uni_df.columns else uni_df.columns[0]
positive_uniprots = set(uni_df[accession_col].astype(str).str.strip())

# --- IDR-level labels ---
idr_df[category_name] = idr_df["uniprot_accession"].isin(positive_uniprots).astype(int)
annotation_df = idr_df[["idr_name", category_name]]

target_file.parent.mkdir(parents=True, exist_ok=True)
annotation_df.to_csv(target_file, sep="\t", index=False)

# --- coverage: how many input proteins have at least one IDR in the feature file? ---
idr_uniprots = set(idr_df["uniprot_accession"])
with_idr = positive_uniprots & idr_uniprots
without_idr = positive_uniprots - idr_uniprots

print(f"Feature file: {features_file}")
print(f"IDRs in feature file: {len(idr_df):,}")
print(f"Proteins in your list: {len(positive_uniprots):,}")
print(f"  with ≥1 IDR in feature file: {len(with_idr):,}")
print(f"  with no IDR in feature file: {len(without_idr):,}")
print(f"IDRs labeled positive (1): {annotation_df[category_name].sum():,}")
print(f"Saved annotation: {target_file}")

fig, ax = plt.subplots(figsize=(PNAS_SINGLE, PNAS_SINGLE * 0.85))
bars = ax.bar(
    ["With IDR(s)\nin feature file", "Without IDR(s)\nin feature file"],
    [len(with_idr), len(without_idr)],
    color=['#2166ac', '#b2b2b2'],
    width=0.5,
    edgecolor='white',
    linewidth=0.5,
)
ax.set_ylabel(f"Proteins in {category_name}")
ax.set_ylim(0, max(len(with_idr), len(without_idr)) * 1.18)
for bar, n in zip(bars, [len(with_idr), len(without_idr)]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + ax.get_ylim()[1] * 0.01,
            f"{n:,}", ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.show()

## Step 3 — Set parameters

Adjust:
- `n_reps` — number of repetitions (default 100; use a smaller value for a quick test)
- `specificity` (default: 0.99) — controls how strict the positive/negative cutoff is within each run. Higher values → fewer false positives and fewer predicted positives. Lower values → more permissive predictions.
- `consistency_cutoff` — consistency_cutoff (default: 0.5) — controls how consistently an IDR must be predicted positive across runs. 1.0 = positive in every run. 0.5 = positive in at least half of the runs. Higher values produce a smaller, more confident set of predictions.

In [ ]:
specificity = 0.99
n_reps = 5
consistency_cutoff = 0.5

## Step 4 — Run FAIDR

Click **Run** on the cell below and wait until both progress bars finish.

| Progress bar | What it does |
|---|---|
| **Step 1/2 — Cross-validation (ROC / AUC)** | Repeated cross-validation to estimate how well the model generalizes. A ROC plot is drawn automatically; its title shows **mean CV AUC** |
| **Step 2/2 — Final training (IDR predictions)** | Repeated training to produce per-IDR probabilities and consistency scores. **`run_summary.tsv` metrics (Step 5) are from this stage — training AUC, not CV** |

In [ ]:
result = run_faidr_cv(
    target_file=target_file,
    feature_type=feature_type,
    features_path=features_path,
    specificity=specificity,
    n_reps=n_reps,
    consistency_cutoff=consistency_cutoff,
    show_progress=True,
)

print("Done. Output folder:", result["out_dir"])

It is important to note that we do not expect perfect ROC AUC values in this setting because the model is expected to partially re-evaluate and refine the original labels based on broader learned IDR patterns.

A mean ROC AUC substantially above approximately in the range of ~0.65–1.0 indicates that the model has learned sequence features that distinguish the positive IDR set from the background. In this context, the goal is not to perfectly reproduce the original protein-level labels, but rather to identify IDRs whose feature profiles are most consistent with the patterns associated with the compartment or functional category of interest.

## Step 5 — Summary metrics

### CV vs training AUC — read this first

In **Step 4**, FAIDR runs two stages:

1. **Cross-validation (Step 1/2)** — the ROC plot shown in the notebook uses **unbiased** predictions on held-out data. Its title reports **mean CV AUC** — this is the main check that the model can be trained properly.
2. **Final training (Step 2/2)** — the model is trained on all IDR labela to produce per-IDR probabilities (`idr_predictions.tsv`). Metrics in `run_summary.tsv` come from **this training stage**, not from CV.

So: **trust the CV ROC plot for validation**; use `run_summary.tsv` for run metadata and **training-set** performance (often optimistic compared to CV).

### Column guide (`run_summary.tsv`)

| Column | What it means |
|--------|----------------|
| **category** | Name of your target (from the annotation filename, e.g. `GO0005730`) |
| **thresh** | Mean probability cutoff across `n_reps` training runs (chosen from each run's ROC curve at your `specificity` setting) |
| **above_thresh_in_annotated** | Mean number of **true positive** IDRs (label 1) with probability above `thresh` per run — roughly “how many known positives were recovered” |
| **above_thresh_not_in_annotated** | Mean number of **true negative** IDRs (label 0) with probability above `thresh` per run — false positives among negatives at that cutoff |
| **IDR_AUC_train** | Mean ROC AUC at **IDR level** on the **training set** of each run (all positives + subsampled negatives). **Not CV AUC.** |
| **Prot_AUC_train** | Mean ROC AUC at **protein level** on the **training set** (one score per protein, aggregated from its IDRs). **Not CV AUC.** |
| **N_annotated_IDRs** | Total IDRs labeled **1** in your annotation file (fixed for this run) |

**Typical reading:** high `IDR_AUC_train` / `Prot_AUC_train` with a lower CV AUC in the Step 4 plot suggests some overfitting to the training sample.

In [ ]:
summary = result["classi_info"].T
summary.columns = ["value"]
summary.index.name = "metric"
display(summary)

print("\nSaved to:", result["out_dir"] / "run_summary.tsv")

## Step 6 — Per-IDR predictions (preview)

Each row is one IDR. Important columns:

- **mean_probability** — average predicted probability across all repetitions
- **consistency_fraction** — fraction of repetitions where the IDR was called positive (above the learned threshold); ranges from 0 to 1

Rows with `consistency_fraction ≥ consistency_cutoff` are saved to `idr_predictions_consistency_<cutoff>.tsv`.

In [ ]:
predictions = result["idr_results"]
filtered = result["idr_results_filtered"]
display(predictions.head(10))

## Step 7 — Venn diagram: annotation vs. consistent FAIDR predictions

We compare two sets of IDRs:

1. **Annotated positives** — IDRs labeled `1` in initial labeling
2. **Consistent predictions** — IDRs from `idr_predictions_consistency_<cutoff>.tsv` (already filtered in Step 2)

The **overlap** shows IDRs that were both annotated and reliably recovered by FAIDR.

In [ ]:
annotation = pd.read_csv(target_file, sep="\t")
label_col = annotation.columns[1]

annotated_positive = set(annotation.loc[annotation[label_col] == 1, "idr_name"])
consistent_predicted = set(result["idr_results_filtered"]["idr_name"])

overlap = annotated_positive & consistent_predicted
only_annotated = annotated_positive - consistent_predicted
only_predicted = consistent_predicted - annotated_positive

fig, ax = plt.subplots(figsize=(PNAS_15COL, PNAS_15COL * 0.88))
v = venn2(
    subsets=(len(only_annotated), len(only_predicted), len(overlap)),
    set_labels=(
        f"Annotated positives\n(n={len(annotated_positive):,})",
        f"Consistent predictions\n(≥{consistency_cutoff}, n={len(consistent_predicted):,})",
    ),
    set_colors=('#2166ac', '#d6604d'),
    alpha=0.45,
    ax=ax,
)
for text in (v.set_labels or []):
    if text: text.set_fontsize(7)
for text in (v.subset_labels or []):
    if text: text.set_fontsize(8)
ax.set_title(f"{label_col}: annotation vs. FAIDR predictions")
plt.tight_layout()
plt.show()

The overlap is incomplete, but this is expected for several reasons:

1. The feature space used by the model may not capture all determinants of condensate association. Some biologically relevant mechanisms may depend on structural features, post-translational modifications, or interaction partners that are not represented in the current feature set.
2. Experimental labels are available only at the protein level, whereas the model operates at the IDR level. Many proteins contain multiple IDRs, and not all of them are necessarily involved in condensate localization. As a result, the initial IDR labels are inherently noisy. During training, the model attempts to identify which IDRs are most consistent with the sequence features associated with the condensate, which can lead to predictions that do not perfectly overlap with the original protein-level assignments.
3. The specificity-based prediction threshold (0.99 in our case) is intentionally stringent. Therefore, the predicted sets contain only the IDRs for which the model has relatively high confidence